# validation check — team archetypes (final) -> LR, HOME market

`goals_foragst` space, home market. Cluster teams into archetypes by their END-OF-SEASON profile
(final), label each match by its (home, away) archetype cell, keep train-profitable cells. Matches
are assigned by running stats (honest at bet time). The archetype-count sweep is an extra check.

In [1]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [2]:
# --- SPACE (modifiable) — the one place that defines this check's space ------
# The running (point-in-time) season features that describe the match. This is the ONLY
# cell to change to move the whole check to another space.
FEATURES = [
    "hmt_season_goals_for_avg",
    "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg",
    "awt_season_goals_agst_avg",
]

In [3]:
# --- TARGET (modifiable) -----------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the market-consensus implied probability
# (mrkt), used for both the profitability check and the ROI — one odds source, never mixed.
OUTCOME = "t_home_flg"          # 1 when the home win happened
IMPLIED = "mrkt_home_impl"      # implied prob = 1 / mrkt odds (profitability + betting)

In [4]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages; drop rows missing any
# feature or the target.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [5]:
# --- SEGMENTER (modifiable) — team archetypes on FINAL (end-of-season) stats -
# Cluster individual TEAMS into archetypes by their END-OF-SEASON per-side profile ("who the team
# really was"), label each match by its (home archetype, away archetype) cell, keep the cells that beat
# the market on train. Boundaries use FINAL; every match is still ASSIGNED by its RUNNING stats, so the
# end-of-season value never locates a match (no leak) — it only shapes where the archetype regions sit.
# A different method (running boundaries, match-level clustering, ...) = a different notebook.

# Attach each team's end-of-season per-side stats (this method's boundaries need them). These mirror
# the running FEATURES above; they feed only the boundaries.
_tsf = pd.read_parquet(f"C:/Users/twluc/frame/01_data/02_features/{GROUP}/team_season_final.parquet")
_tsf = _tsf[["team", "season", "season_goals_for_avg", "season_goals_agst_avg"]]
matches = matches.merge(_tsf.rename(columns={"team": "hmt_name",
                                             "season_goals_for_avg": "hf", "season_goals_agst_avg": "ha"}),
                        on=["hmt_name", "season"], how="left")
matches = matches.merge(_tsf.rename(columns={"team": "awt_name",
                                             "season_goals_for_avg": "af", "season_goals_agst_avg": "aa"}),
                        on=["awt_name", "season"], how="left")

HOME_RUN, AWAY_RUN = FEATURES[:2], FEATURES[2:]   # running per-side 2D (used to ASSIGN each match)
HOME_FIN, AWAY_FIN = ["hf", "ha"], ["af", "aa"]   # final per-side 2D (used to fit the archetypes)

# Keep the clusters/cells where the target happened MORE often than the market priced it, on the TRAIN
# fold only, so the held-out fold never influences the choice.
def keep_profitable(train, labels, min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    return set(stats.index[(stats["n"] >= min_matches) & (stats["won"] > stats["implied"])])

def team_grid_final_segmenter(n_types, min_matches=100):
    def segmenter(train):
        pts = np.vstack([train[HOME_FIN].to_numpy(), train[AWAY_FIN].to_numpy()])   # archetypes from FINAL
        km = KMeans(n_clusters=n_types, random_state=0, n_init=10).fit(pts)
        def cells(rows):   # a match's cell = "home_type-away_type", assigned by RUNNING per-side stats
            h = km.predict(rows[HOME_RUN].to_numpy())
            a = km.predict(rows[AWAY_RUN].to_numpy())
            return pd.Series([f"{i}-{j}" for i, j in zip(h, a)], index=rows.index)
        kept = keep_profitable(train, cells(train), min_matches)
        def labeler(rows):
            c = cells(rows)
            return c.where(c.isin(kept))     # non-kept cells -> NaN -> not bet
        return labeler
    return segmenter

N = 3   # team archetypes -> N**2 grid cells (see the sweep extra check below)
segmenter = team_grid_final_segmenter(N)

# The segmenter's self-description: fit archetypes on all matches (in-sample) and show each (home, away)
# cell's edge — size, outcome rate vs implied, ROI. In-sample: where the edge sits, not gated evidence.
_km = KMeans(n_clusters=N, random_state=0, n_init=10).fit(np.vstack([matches[HOME_FIN].to_numpy(), matches[AWAY_FIN].to_numpy()]))
print("archetype centres (for, agst): " + "  ".join(f"{i}:[{c[0]:.2f} {c[1]:.2f}]" for i, c in enumerate(_km.cluster_centers_)))
_cell = pd.Series([f"{i}-{j}" for i, j in zip(_km.predict(matches[HOME_RUN].to_numpy()), _km.predict(matches[AWAY_RUN].to_numpy()))], index=matches.index)
print(f"segment edges (in-sample, n={N} archetypes -> {N*N} cells):")
print(" cell      n   rate%  impl%   edge     ROI   keep")
_rows = []
for cell, m in matches.groupby(_cell):
    dr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    _rows.append((cell, len(m), dr, im, dr - im, pooled_roi(m[IMPLIED], m[OUTCOME].astype(int))))
for cell, n, dr, im, ed, roi in sorted(_rows, key=lambda r: -r[5]):
    print(f" {cell:>5} {n:>7}  {dr*100:5.1f}  {im*100:5.1f}  {ed*100:+5.1f}  {roi:+6.3f}  {'KEEP' if ed > 0 else ''}")

archetype centres (for, agst): 0:[1.05 1.87]  1:[2.02 1.14]  2:[1.27 1.33]
segment edges (in-sample, n=3 archetypes -> 9 cells):
 cell      n   rate%  impl%   edge     ROI   keep
   1-1     401   51.9   46.9   +5.0  +0.107  KEEP
   2-1     657   28.8   29.1   -0.3  -0.010  
   1-2     631   63.1   65.3   -2.2  -0.034  
   2-2    1194   43.7   45.7   -2.0  -0.043  
   2-0     666   53.2   56.4   -3.3  -0.058  
   1-0     425   70.1   74.7   -4.6  -0.061  
   0-0     453   41.7   45.6   -3.9  -0.086  
   0-1     441   18.8   21.7   -2.8  -0.131  
   0-2     665   28.0   34.5   -6.5  -0.190  


In [6]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: any sklearn-style estimator with .fit(X, y) and .predict_proba(X).
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Summary
The three gates at a glance, and the one-line row written to `model_checks/results.md`.

In [7]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="goals_foragst_home_kmeans_team_lr", space="goals_foragst", target="home",
              abt_filter="major, gn>8, dropna", segmentation=f"KMeans n={N} team archetypes (final) -> {N}x{N} grid",
              segment_filter="train-profitable cells")

goals_foragst_home_kmeans_team_lr   pooled -0.091 (474 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           FAIL  (+1/4 seasons)
    2223 +0.187    2324 -0.163    2425 -0.124    2526 -0.091

GATE 2  league — ROI by league / drop best   FAIL
    france +0.168    england -0.083    italy -0.109    germany -0.212    spain -0.227
    pooled without best league (france): -0.148

GATE 3  walk-forward — ROI by test season    FAIL
    2425 -0.089    2526 -0.091


## Extra checks — below the template
Everything above is the standard template (config cells + Summary). Cells below are notebook-specific
diagnostics, outside the template — they may use knobs a given segmenter has (e.g. a cluster count)
that other methods do not.

In [8]:
# EXTRA CHECK (below the template) — knob robustness: sweep the number of team archetypes = 2..8
# (grid cells = n**2). Recomputes the whole honest per-fold pipeline for each n. A real edge survives a
# range of n; one that appears at a single n is fragile.
print(" n  cells   pooled     bets   seasons+")
for n in range(2, 9):
    b = collect_bets(matches, season_folds(matches), features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=team_grid_final_segmenter(n), make_model=make_model, buffer=BUFFER)
    ps = roi_grouped(b, "fold")
    print(f" {n}  {n*n:>4}   {pooled_roi(b['implied'], b['won']):+.4f}   {len(b):>4}   +{int((ps > 0).sum())}/{len(ps)}")

 n  cells   pooled     bets   seasons+
 2     4   -0.1304    306   +0/3
 3     9   -0.0906    474   +1/4
 4    16   -0.0135    760   +1/4
 5    25   -0.0549    793   +1/4
 6    36   +0.0313    498   +2/4
 7    49   +0.0579    340   +2/4
 8    64   -0.0426    219   +1/4
